# 04 — Statistical Analysis


Includes central tendency, spread, shape, confidence-oriented normality tests and correlations.


In [1]:
from pathlib import Path
import sys, pandas as pd
ROOT=Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
DATA=ROOT/'data'/'uploads'/'sample_business_data.csv'
df=pd.read_csv(DATA)
df.head()

from src.analysis.core import statistical_summary
statistical_summary(df)


,count,mean,std,min,25%,50%,75%,max,variance,skewness,kurtosis,normality_p_value
customer_id,500.0,250.5000,144.481833,1.0,125.75,250.5,375.250,500.0,2.087500e+04,0.000000,-1.200000,1.169302e-69
age,500.0,43.8760,14.855504,18.0,32.00,43.0,56.000,70.0,2.206860e+02,0.009557,-1.127063,7.427140e-44
income,500.0,85746.1120,38260.399351,18106.0,53008.25,88086.0,119471.000,149542.0,1.463858e+09,-0.082677,-1.214633,3.287593e-77
visits,500.0,10.3640,5.769756,1.0,5.00,10.0,15.000,20.0,3.329008e+01,0.035133,-1.183462,4.338579e-62
satisfaction,500.0,3.0408,1.185304,1.0,2.00,3.1,4.025,5.0,1.404945e+00,-0.020171,-1.247280,1.818160e-99
purchased,500.0,0.6080,0.488686,0.0,0.00,1.0,1.000,1.0,2.388136e-01,-0.443777,-1.810319,0.000000e+00


In [2]:
from scipy import stats
num=df.select_dtypes('number')
num.corr()


,customer_id,age,income,visits,satisfaction,purchased
customer_id,1.000000,0.051662,-0.057705,-0.070900,-0.008342,-0.037380
age,0.051662,1.000000,0.006254,-0.029493,-0.087039,-0.089247
income,-0.057705,0.006254,1.000000,0.009109,0.008549,0.397630
visits,-0.070900,-0.029493,0.009109,1.000000,0.037237,0.368409
satisfaction,-0.008342,-0.087039,0.008549,0.037237,1.000000,0.330046
purchased,-0.037380,-0.089247,0.397630,0.368409,0.330046,1.000000


In [ ]:
#Loading Cleanded Data

from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

ROOT = Path.cwd().parent

CLEANED_FILE = ROOT / "artifacts" / "cleaned_data.parquet"
df = pd.read_parquet(CLEANED_FILE)

# Identifier columns should not be analysed as measurements.
ID_COLUMNS = ["customer_id"]

analysis_df = df.drop(
    columns=[column for column in ID_COLUMNS if column in df.columns]
)

print("Dataset shape:", df.shape)
print("Analysis shape:", analysis_df.shape)

display(analysis_df.head())

Dataset shape: (500, 7)
Analysis shape: (500, 6)


,age,income,region,visits,satisfaction,purchased
0,58,47184,North,9,2.0,0
1,65,44868,North,19,2.7,0
2,23,75314,South,17,3.4,1
3,63,127974,South,15,3.4,1
4,66,59853,West,11,2.1,0


In [ ]:
#Variable Classification

numeric_columns = analysis_df.select_dtypes(
    include=np.number
).columns.tolist()

categorical_columns = analysis_df.select_dtypes(
    exclude=np.number
).columns.tolist()

# purchased is numerically stored but represents a category.
TARGET = "purchased"

if TARGET in numeric_columns:
    numeric_columns.remove(TARGET)

if TARGET in analysis_df.columns and TARGET not in categorical_columns:
    categorical_columns.append(TARGET)

print("Continuous/numeric variables:", numeric_columns)
print("Categorical variables:", categorical_columns)


Continuous/numeric variables: ['age', 'income', 'visits', 'satisfaction']
Categorical variables: ['region', 'purchased']


In [5]:
#Descriptive STatistics


descriptive_statistics = analysis_df[numeric_columns].describe().T

descriptive_statistics["variance"] = (
    analysis_df[numeric_columns].var()
)

descriptive_statistics["range"] = (
    analysis_df[numeric_columns].max()
    - analysis_df[numeric_columns].min()
)

descriptive_statistics["iqr"] = (
    analysis_df[numeric_columns].quantile(0.75)
    - analysis_df[numeric_columns].quantile(0.25)
)

descriptive_statistics["skewness"] = (
    analysis_df[numeric_columns].skew()
)

descriptive_statistics["kurtosis"] = (
    analysis_df[numeric_columns].kurtosis()
)

display(descriptive_statistics.round(3))

,count,mean,std,min,25%,50%,75%,max,variance,range,iqr,skewness,kurtosis
age,500.0,43.876,14.856,18.0,32.00,43.0,56.000,70.0,2.206860e+02,52.0,24.000,0.010,-1.127
income,500.0,85746.112,38260.399,18106.0,53008.25,88086.0,119471.000,149542.0,1.463858e+09,131436.0,66462.750,-0.083,-1.215
visits,500.0,10.364,5.770,1.0,5.00,10.0,15.000,20.0,3.329000e+01,19.0,10.000,0.035,-1.183
satisfaction,500.0,3.041,1.185,1.0,2.00,3.1,4.025,5.0,1.405000e+00,4.0,2.025,-0.020,-1.247


In [6]:
#Mode and categorical frequencies

for column in categorical_columns:
    print(f"\n{'=' * 60}")
    print(f"Column: {column}")
    print(f"Mode: {analysis_df[column].mode().iloc[0]}")
    
    frequency_table = (
        analysis_df[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="frequency")
    )
    
    frequency_table["percentage"] = (
        frequency_table["frequency"]
        / len(analysis_df)
        * 100
    ).round(2)
    
    display(frequency_table)


Column: region
Mode: West


,region,frequency,percentage
0,West,134,26.8
1,East,127,25.4
2,North,124,24.8
3,South,115,23.0



Column: purchased
Mode: 1


,purchased,frequency,percentage
0,1,304,60.8
1,0,196,39.2


In [7]:
# 95% confidence Intervals

confidence_results = []

for column in numeric_columns:
    values = analysis_df[column].dropna()

    count = len(values)
    mean = values.mean()
    standard_error = stats.sem(values)

    confidence_interval = stats.t.interval(
        confidence=0.95,
        df=count - 1,
        loc=mean,
        scale=standard_error,
    )

    confidence_results.append({
        "column": column,
        "sample_size": count,
        "mean": mean,
        "standard_error": standard_error,
        "ci_95_lower": confidence_interval[0],
        "ci_95_upper": confidence_interval[1],
    })

confidence_df = pd.DataFrame(confidence_results)

display(confidence_df.round(3))

,column,sample_size,mean,standard_error,ci_95_lower,ci_95_upper
0,age,500,43.876,0.664,42.571,45.181
1,income,500,85746.112,1711.057,82384.348,89107.876
2,visits,500,10.364,0.258,9.857,10.871
3,satisfaction,500,3.041,0.053,2.937,3.145


In [8]:
#Normality Tests  

normality_results = []

for column in numeric_columns:
    values = analysis_df[column].dropna()

    statistic, p_value = stats.normaltest(values)

    normality_results.append({
        "column": column,
        "test": "D'Agostino-Pearson",
        "statistic": statistic,
        "p_value": p_value,
        "is_normal_at_0.05": p_value > 0.05,
    })

normality_df = pd.DataFrame(normality_results)

display(normality_df.round(6))

,column,test,statistic,p_value,is_normal_at_0.05
0,age,D'Agostino-Pearson,198.617207,0.0,False
1,income,D'Agostino-Pearson,352.217793,0.0,False
2,visits,D'Agostino-Pearson,282.585458,0.0,False
3,satisfaction,D'Agostino-Pearson,454.716198,0.0,False


In [9]:
#Pearlson Correlation

pearson_correlation = analysis_df[numeric_columns].corr(
    method="pearson"
)

display(pearson_correlation.round(3))

,age,income,visits,satisfaction
age,1.000,0.006,-0.029,-0.087
income,0.006,1.000,0.009,0.009
visits,-0.029,0.009,1.000,0.037
satisfaction,-0.087,0.009,0.037,1.000


In [10]:
# Spearman correlation

spearman_correlation = analysis_df[numeric_columns].corr(
    method="spearman"
)

display(spearman_correlation.round(3))

,age,income,visits,satisfaction
age,1.000,0.006,-0.027,-0.087
income,0.006,1.000,0.007,0.004
visits,-0.027,0.007,1.000,0.038
satisfaction,-0.087,0.004,0.038,1.000


In [11]:
# point biserial relationship between variables

target_relationships = []

for column in numeric_columns:
    group_zero = analysis_df.loc[
        analysis_df[TARGET] == 0,
        column
    ].dropna()

    group_one = analysis_df.loc[
        analysis_df[TARGET] == 1,
        column
    ].dropna()

    correlation, correlation_p = stats.pointbiserialr(
        analysis_df[TARGET],
        analysis_df[column],
    )

    t_statistic, t_test_p = stats.ttest_ind(
        group_zero,
        group_one,
        equal_var=False,
    )

    target_relationships.append({
        "feature": column,
        "target_0_mean": group_zero.mean(),
        "target_1_mean": group_one.mean(),
        "point_biserial_correlation": correlation,
        "correlation_p_value": correlation_p,
        "welch_t_statistic": t_statistic,
        "welch_t_test_p_value": t_test_p,
        "significant_at_0.05": t_test_p < 0.05,
    })

target_relationship_df = pd.DataFrame(target_relationships)

display(target_relationship_df.round(6))

,feature,target_0_mean,target_1_mean,point_biserial_correlation,correlation_p_value,welch_t_statistic,welch_t_test_p_value,significant_at_0.05
0,age,45.525510,42.812500,-0.089247,0.046085,1.989507,0.047311,True
1,income,66818.224490,97949.618421,0.397630,0.000000,-9.632616,0.000000,True
2,visits,7.719388,12.069079,0.368409,0.000000,-8.975040,0.000000,True
3,satisfaction,2.554082,3.354605,0.330046,0.000000,-8.047435,0.000000,True


In [13]:
# Region vs purchased relationship

contingency_table = pd.crosstab(
    analysis_df["region"],
    analysis_df[TARGET],
)

chi2, p_value, degrees_of_freedom, expected = (
    stats.chi2_contingency(contingency_table)
)

expected_df = pd.DataFrame(
    expected,
    index=contingency_table.index,
    columns=contingency_table.columns,
)

print("Observed frequencies:")
display(contingency_table)

print("Expected frequencies:")
display(expected_df.round(2))

print(f"Chi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom: {degrees_of_freedom}")
print(f"P-value: {p_value:.6f}")
print(f"Significant association: {p_value < 0.05}")

Observed frequencies:


purchased,0,1
region,,
East,53,74
North,51,73
South,44,71
West,48,86


Expected frequencies:


purchased,0,1
region,,
East,49.78,77.22
North,48.61,75.39
South,45.08,69.92
West,52.53,81.47


Chi-square statistic: 1.2198
Degrees of freedom: 3
P-value: 0.748253
Significant association: False


In [14]:
# Effect size (Cramér's V)

sample_size = contingency_table.to_numpy().sum()

minimum_dimension = min(contingency_table.shape) - 1

cramers_v = np.sqrt(
    chi2 / (sample_size * minimum_dimension)
)

print(f"Cramér's V: {cramers_v:.4f}")

Cramér's V: 0.0494


In [15]:
# Save statistical results

STATISTICS_DIR = ROOT / "artifacts" / "statistics"
STATISTICS_DIR.mkdir(parents=True, exist_ok=True)

descriptive_statistics.to_csv(
    STATISTICS_DIR / "descriptive_statistics.csv"
)

confidence_df.to_csv(
    STATISTICS_DIR / "confidence_intervals.csv",
    index=False,
)

normality_df.to_csv(
    STATISTICS_DIR / "normality_tests.csv",
    index=False,
)

pearson_correlation.to_csv(
    STATISTICS_DIR / "pearson_correlation.csv"
)

spearman_correlation.to_csv(
    STATISTICS_DIR / "spearman_correlation.csv"
)

target_relationship_df.to_csv(
    STATISTICS_DIR / "target_relationship_tests.csv",
    index=False,
)

contingency_table.to_csv(
    STATISTICS_DIR / "region_target_contingency.csv"
)

print("Statistical results saved successfully.")

Statistical results saved successfully.
